In [7]:
!tar -xzvf /content/ensemble_task_3_data_oct24-oct25.tar.gz

data.csv
devices.csv


In [47]:
import pandas as pd
import numpy as np

In [23]:
results = []

df_devices = pd.read_csv('devices.csv')


In [30]:
from tqdm import tqdm

path_data =  "data.csv"

keep_columns = ['deviceId', 'timedate', 't1', 't2', 't7', 't8','t9', 'x1', 'x2', 'x3', 'deviceType']

all_sampled_chunks = []

reader = pd.read_csv(path_data, chunksize=1000000, usecols=keep_columns)

for chunk in tqdm(reader, desc="Slicing co 12 rekordów (co 1h)"):
    sampled = chunk.iloc[::12].copy()
    sampled['timedate'] = pd.to_datetime(sampled['timedate'])

    all_sampled_chunks.append(sampled)

df_data_sampled = pd.concat(all_sampled_chunks, ignore_index=True)

print(f"Pomyślnie wczytano {len(df_data_sampled)} rekordów.")


Slicing co 12 rekordów (co 1h): 65it [05:31,  5.10s/it]


Pomyślnie wczytano 5374819 rekordów.


In [32]:
df_merged = df_devices.merge(df_data_sampled, on='deviceId')

In [36]:
df_merged['year'] = df_merged['timedate'].dt.year

In [38]:
df_merged['month'] = df_merged['timedate'].dt.month

In [40]:
df_merged['hour'] = df_merged['timedate'].dt.hour

In [43]:
df_merged.drop('timedate', axis=1, inplace=True)

In [44]:
df_merged

,latitude,longitude,deviceId,t1,t2,t7,t8,t9,x1,x2,x3,deviceType,year,month,hour
0,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.29,0.05,0.20,0.51,0.42,0.0,0.000000,8,19,2024,10,0
1,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.29,0.05,0.20,0.54,0.40,0.0,0.579203,8,19,2024,10,1
2,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.29,0.05,0.20,0.49,0.39,0.0,0.000000,8,19,2024,10,2
3,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.29,0.05,0.21,0.52,0.42,0.0,0.269766,8,19,2024,10,3
4,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.28,0.05,0.21,0.54,0.40,0.0,0.579203,8,19,2024,10,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5374814,52.5,15.3,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,0.29,0.05,0.21,0.48,0.35,0.0,NaN,8,11,2025,10,19
5374815,52.5,15.3,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,0.29,0.05,0.21,0.47,0.35,0.0,NaN,8,11,2025,10,20
5374816,52.5,15.3,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,0.29,0.05,0.21,0.47,0.35,0.0,NaN,8,11,2025,10,21
5374817,52.5,15.3,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,0.29,0.05,0.21,0.46,0.35,0.0,NaN,8,11,2025,10,22


In [46]:
# device + warunki pracy → średnie zużycie energii w miesiącu
# średnia temperatura
# minimalna temperatura
# max temperatura
# średnia częstotliwość pracy urządzenia

'''
t1 Temperatura zewnętrzna Temperatura otoczenia na zewnątrz; min.-maks.
t2 Temperatura wewnętrzna Temperatura otoczenia wewnątrz; min.-maks.
t3 Temperatura HEX źródła 1 Temperatura 1 na wymienniku ciepła po stronie źródła; min.-maks. znormalizowana.
t4 Temperatura HEX źródła 2 Temperatura 2 na wymienniku ciepła po stronie źródła; min.-maks. znormalizowana.
t5 Temperatura HEX obciążenia 1 Temperatura 1 na wymienniku ciepła po stronie odbioru; min.-maks. znormalizowana.
t6 Temperatura HEX obciążenia 2 Temperatura 2 na wymienniku ciepła po stronie odbioru; min.-maks. znormalizowana.
t7 Temperatura zasobnika CWU Temperatura zasobnika ciepłej wody użytkowej; min.-maks. znormalizowana.
t8 Temperatura obiegu chłodzenia 1 Temperatura 1 obiegu chłodzenia; min.-maks. znormalizowana.
t9 Temperatura bufora CO Temperatura bufora centralnego ogrzewania; min.-maks. znormalizowana.
t10 Temperatura powietrza źródła HEX 1 Temperatura powietrza 1 na wymienniku ciepła po stronie źródła

; znormalizowana min.-maks.
t11 Temperatura powietrza źródła HEX 2 Temperatura powietrza 2 na wymienniku ciepła po stronie źródła

; znormalizowana min.-maks.
t12 Temperatura obciążenia HEX 3 Temperatura 3 na wymienniku ciepła po stronie odbiornika

; znormalizowana min.-maks.
t13 Temperatura obciążenia HEX 4 Temperatura 4 na wymienniku ciepła po stronie odbiornika

znorm min-max
x1 - czestotliwosc pracy urzadzenia
x2 Wskaźnik obciążenia sieci Indywidualny wskaźnik obciążenia sieci elektrycznej (cel prognozy).
x3 Typ krzywej grzewczej Typ konfiguracji krzywej grzewczej używanej przez
urządzenie.
deviceType Typ urządzenia Kategoria urządzenia zakodowana w liczbach całkowitych
'''
features = {
    "x1": ["mean", "max"],
    "t1": ["mean", "min", "max"],
    "t2": ["mean"],
    "t7": ["mean"],
    "t8": ["mean"],
    "t9": ["mean"]
}
# roznice temperatur
df_merged["temp_diff_ext_int"] = df_merged["t1"] - df_merged["t2"]
df_merged["delta_source"] = df_merged["t7"] - df_merged["t8"]
df_merged["delta_load"] = df_merged["t8"] - df_merged["t9"]





In [49]:
df_merged["month_sin"] = np.sin(2*np.pi*df_merged["month"]/12)
df_merged["month_cos"] = np.cos(2*np.pi*df_merged["month"]/12)

In [50]:
# praca pompy jest mocniejsza gdy jest zima
df_merged["heat_intensity"] = df_merged["x1"] * (1 - df_merged["t1"])

In [54]:
# lokalizacja
df_merged["north"] = df_merged["latitude"] > df_merged["latitude"].median()

In [56]:
# licba pomiarow w miesiacu
df_merged["n_measurements"] = df_merged.groupby(["deviceId","year","month"])["x2"].transform("count")

In [57]:
df_merged

,latitude,longitude,deviceId,t1,t2,t7,t8,t9,x1,x2,...,hour,temp_diff_ext_int,delta_source,delta_load,month_sin,month_cos,heat_intensity,estimated_baseload,north,n_measurements
0,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.29,0.05,0.20,0.51,0.42,0.0,0.000000,...,0,0.24,-0.31,0.09,-0.866025,0.5,0.0,0.0,False,743
1,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.29,0.05,0.20,0.54,0.40,0.0,0.579203,...,1,0.24,-0.34,0.14,-0.866025,0.5,0.0,0.0,False,743
2,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.29,0.05,0.20,0.49,0.39,0.0,0.000000,...,2,0.24,-0.29,0.10,-0.866025,0.5,0.0,0.0,False,743
3,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.29,0.05,0.21,0.52,0.42,0.0,0.269766,...,3,0.24,-0.31,0.10,-0.866025,0.5,0.0,0.0,False,743
4,50.0,18.3,000cc3cb7f030c3d0c481bd0e7cf42ee283012c3cb5bfc...,0.28,0.05,0.21,0.54,0.40,0.0,0.579203,...,4,0.23,-0.33,0.14,-0.866025,0.5,0.0,0.0,False,743
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5374814,52.5,15.3,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,0.29,0.05,0.21,0.48,0.35,0.0,NaN,...,19,0.24,-0.27,0.13,-0.866025,0.5,0.0,0.0,True,0
5374815,52.5,15.3,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,0.29,0.05,0.21,0.47,0.35,0.0,NaN,...,20,0.24,-0.26,0.12,-0.866025,0.5,0.0,0.0,True,0
5374816,52.5,15.3,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,0.29,0.05,0.21,0.47,0.35,0.0,NaN,...,21,0.24,-0.26,0.12,-0.866025,0.5,0.0,0.0,True,0
5374817,52.5,15.3,ffd5e815486d2b094a0ff197111e96c998e11fda1626f7...,0.29,0.05,0.21,0.46,0.35,0.0,NaN,...,22,0.24,-0.25,0.11,-0.866025,0.5,0.0,0.0,True,0


In [63]:
df_merged.drop('estimated_baseload', axis=1, inplace=True)

In [64]:
df_merged.drop('heat_intensity', axis=1, inplace=True)

In [65]:
df_merged.columns

Index(['latitude', 'longitude', 'deviceId', 't1', 't2', 't7', 't8', 't9', 'x1',
       'x2', 'x3', 'deviceType', 'year', 'month', 'hour', 'temp_diff_ext_int',
       'delta_source', 'delta_load', 'month_sin', 'month_cos', 'north',
       'n_measurements'],
      dtype='object')

In [67]:
# agregacja miesieczna
agg_dict = {
    "x2": "mean",
    "x1": "mean",
    "t1": "mean",
    "t2": "mean",
    "t7": "mean",
    "t8": "mean",
    "t9": "mean",
    "temp_diff_ext_int": "mean",
    "delta_source": "mean",
    "delta_load": "mean",
    "n_measurements": "sum",
    "latitude": "first",
    "longitude": "first",
    "deviceType": "first"
}

df_monthly = (
    df_merged
    .groupby(["deviceId","year","month"])
    .agg(agg_dict)
    .reset_index()
)

In [71]:
df_model = df_monthly.dropna(subset=['x2'])

In [72]:
target = "x2"

features = [
    "x1",
    "t1",
    "t2",
    "t7",
    "t8",
    "t9",
    "temp_diff_ext_int",
    "delta_source",
    "delta_load",
    "latitude",
    "longitude",
    "deviceType",
    "month"
]

X = df_monthly[features]
y = df_monthly[target]

In [73]:
train = df_model[df_model["year"] == 2024]
valid = df_model[df_model["year"] == 2025]

X_train = train[features]
y_train = train[target]

X_valid = valid[features]
y_valid = valid[target]

In [74]:
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05
)

model.fit(X_train, y_train)

pred = model.predict(X_valid)

print("MAE:", mean_absolute_error(y_valid, pred))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000195 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2192
[LightGBM] [Info] Number of data points in the train set: 1776, number of used features: 12
[LightGBM] [Info] Start training from score 0.148686
MAE: 0.03304834777621106
